In [2]:
import sys
import os

# src/ modülünü import edebilmek için
sys.path.insert(0, os.path.abspath(".."))

from src.spark_session import get_spark_session

spark = get_spark_session(
    app_name="Bronze-Streaming",
    master="local[2]",
    driver_memory="3g"
)

print(f"✅ Spark sürümü: {spark.version}")
print(f"✅ Spark UI: http://localhost:4040")

✅ Spark sürümü: 3.5.1
✅ Spark UI: http://localhost:4040


# 🥉 Bronze Katmanı: Kafka'dan Streaming Veri Okuma

Bu notebook Faz 4'ün ilk yarısını oluşturuyor: Kafka'dan gelen JSON mesajları
**şema tanımlayarak** parse edip, **Delta Lake Bronze** tablosuna yazar.

## Bronze Katmanı Felsefesi

- ✅ Veriyi **olduğu gibi** sakla (raw data)
- ❌ Hiç temizleme yapma (null bırak, duplikatları sakla)
- ✅ İz bırak: Kafka'dan gelen `topic`, `partition`, `offset`, `timestamp` bilgilerini de sakla

Bu sayede ileride bir hata çıkarsa veya yeni bir analiz yapmak gerekirse, **ham veriye geri dönebiliriz**.

In [3]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, LongType
)

# Producer'ın gönderdiği JSON formatına uygun şema
message_schema = StructType([
    # Hocanın istediği zorunlu alanlar
    StructField("timestamp", StringType(), True),
    StructField("kullanici_ID", LongType(), True),
    StructField("olay_tipi", StringType(), True),
    StructField("ilgili_ID", StringType(), True),
    
    # Ek alanlar
    StructField("stock_code", StringType(), True),
    StructField("description", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("country", StringType(), True),
    StructField("invoice_date", StringType(), True),
])

print("✅ Şema tanımlandı:")
for field in message_schema.fields:
    print(f"   {field.name}: {field.dataType.simpleString()} (nullable={field.nullable})")

✅ Şema tanımlandı:
   timestamp: string (nullable=True)
   kullanici_ID: bigint (nullable=True)
   olay_tipi: string (nullable=True)
   ilgili_ID: string (nullable=True)
   stock_code: string (nullable=True)
   description: string (nullable=True)
   quantity: int (nullable=True)
   unit_price: double (nullable=True)
   country: string (nullable=True)
   invoice_date: string (nullable=True)


In [4]:
KAFKA_BOOTSTRAP = "localhost:9094"   # host'tan kafka container'a
KAFKA_TOPIC = "online-retail-transactions"

# Streaming okuma
df_kafka_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP) \
    .option("subscribe", KAFKA_TOPIC) \
    .option("startingOffsets", "earliest") \
    .option("failOnDataLoss", "false") \
    .load()

print("✅ Kafka stream tanımlandı (henüz başlatılmadı)")
print("\nKafka raw şeması:")
df_kafka_raw.printSchema()

✅ Kafka stream tanımlandı (henüz başlatılmadı)

Kafka raw şeması:
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [5]:
from pyspark.sql.functions import col, from_json, to_timestamp, current_timestamp

# Kafka'nın value'su binary JSON, string'e cast edip parse et
df_parsed = df_kafka_raw.select(
    # Kafka metadata
    col("topic").alias("kafka_topic"),
    col("partition").alias("kafka_partition"),
    col("offset").alias("kafka_offset"),
    col("timestamp").alias("kafka_timestamp"),
    
    # JSON value'yu parse et
    from_json(col("value").cast("string"), message_schema).alias("data")
).select(
    "kafka_topic", "kafka_partition", "kafka_offset", "kafka_timestamp",
    "data.*",                           # parse edilmiş alanları aç
    current_timestamp().alias("ingestion_time")  # Bronze'a yazılma zamanı
)

print("✅ Bronze stream şeması:")
df_parsed.printSchema()

✅ Bronze stream şeması:
root
 |-- kafka_topic: string (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- kullanici_ID: long (nullable = true)
 |-- olay_tipi: string (nullable = true)
 |-- ilgili_ID: string (nullable = true)
 |-- stock_code: string (nullable = true)
 |-- description: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- country: string (nullable = true)
 |-- invoice_date: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)



In [6]:
# Geçici olarak memory'ye yaz (sadece test için, kalıcı değil)
test_query = df_parsed.writeStream \
    .format("memory") \
    .queryName("bronze_preview") \
    .outputMode("append") \
    .start()

print("✅ Memory stream başladı, 15 saniye bekle...")
import time
time.sleep(15)

# Sorgu sonucunu görüntüle
spark.sql("SELECT * FROM bronze_preview LIMIT 5").show(truncate=False)

# Toplam kaç mesaj geldi?
count = spark.sql("SELECT COUNT(*) as cnt FROM bronze_preview").collect()[0]["cnt"]
print(f"\n📊 Memory'ye toplam {count:,} mesaj yazıldı")

# Stream'i durdur
test_query.stop()
print("✅ Test stream durduruldu")

26/05/10 13:55:14 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/2g/grrxrf7n2sb722s483skjgqm0000gn/T/temporary-09610f4b-2a2d-4562-ab97-0e62b3187bb8. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/10 13:55:14 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


✅ Memory stream başladı, 15 saniye bekle...
+--------------------------+---------------+------------+-----------------------+--------------------------+------------+---------+---------+----------+-----------------------------------+--------+----------+--------------+-------------------+-----------------------+
|kafka_topic               |kafka_partition|kafka_offset|kafka_timestamp        |timestamp                 |kullanici_ID|olay_tipi|ilgili_ID|stock_code|description                        |quantity|unit_price|country       |invoice_date       |ingestion_time         |
+--------------------------+---------------+------------+-----------------------+--------------------------+------------+---------+---------+----------+-----------------------------------+--------+----------+--------------+-------------------+-----------------------+
|online-retail-transactions|1              |0           |2026-05-09 21:12:19.98 |2026-05-09T18:12:19.979810|18102       |purchase |489438   |84032B    |

In [7]:
BRONZE_PATH = os.path.abspath("../delta_lake/bronze/transactions")
CHECKPOINT_PATH = os.path.abspath("../delta_lake/_checkpoints/bronze_transactions")

print(f"📁 Bronze path: {BRONZE_PATH}")
print(f"📁 Checkpoint:  {CHECKPOINT_PATH}")

# Streaming write to Delta
bronze_query = df_parsed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .option("path", BRONZE_PATH) \
    .trigger(processingTime="5 seconds") \
    .start()

print("✅ Bronze streaming başladı!")
print(f"   Stream ID: {bronze_query.id}")
print(f"   Stream durumu: {bronze_query.status}")

📁 Bronze path: /Users/tunakomur/Desktop/big-data-online-retail-pipeline/delta_lake/bronze/transactions
📁 Checkpoint:  /Users/tunakomur/Desktop/big-data-online-retail-pipeline/delta_lake/_checkpoints/bronze_transactions
✅ Bronze streaming başladı!
   Stream ID: 7ecd1eb9-2132-4166-9445-c026522ef490
   Stream durumu: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


26/05/10 13:56:01 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [8]:
import time

print("📊 Stream izleniyor (30 saniye)...\n")

for i in range(6):  # 6 x 5 = 30 saniye
    time.sleep(5)
    
    # Stream metrikleri
    progress = bronze_query.lastProgress
    if progress:
        batch_id = progress.get("batchId", "?")
        num_input = progress.get("numInputRows", 0)
        input_rate = progress.get("inputRowsPerSecond", 0)
        process_time = progress.get("durationMs", {}).get("triggerExecution", 0)
        
        print(f"⏱️  T+{(i+1)*5}s | Batch #{batch_id} | "
              f"Bu batch: {num_input} satır | "
              f"Hız: {input_rate:.1f} satır/s | "
              f"Süre: {process_time}ms")
    else:
        print(f"⏱️  T+{(i+1)*5}s | Henüz batch yok")

print("\n✅ İzleme tamamlandı, stream çalışmaya devam ediyor")

📊 Stream izleniyor (30 saniye)...

⏱️  T+5s | Batch #1 | Bu batch: 0 satır | Hız: 0.0 satır/s | Süre: 5ms
⏱️  T+10s | Batch #1 | Bu batch: 0 satır | Hız: 0.0 satır/s | Süre: 16ms
⏱️  T+15s | Batch #1 | Bu batch: 0 satır | Hız: 0.0 satır/s | Süre: 16ms
⏱️  T+20s | Batch #1 | Bu batch: 0 satır | Hız: 0.0 satır/s | Süre: 16ms
⏱️  T+25s | Batch #1 | Bu batch: 0 satır | Hız: 0.0 satır/s | Süre: 11ms
⏱️  T+30s | Batch #1 | Bu batch: 0 satır | Hız: 0.0 satır/s | Süre: 11ms

✅ İzleme tamamlandı, stream çalışmaya devam ediyor


In [9]:
# Stream'i durdur
bronze_query.stop()
print("✅ Bronze stream durduruldu\n")

# Şimdi Bronze tabloyu batch olarak oku
df_bronze = spark.read.format("delta").load(BRONZE_PATH)
total_rows = df_bronze.count()
print(f"📊 Bronze tabloda toplam: {total_rows:,} satır\n")

# İlk 5 satır
print("İlk 5 satır:")
df_bronze.select("kullanici_ID", "olay_tipi", "ilgili_ID", "quantity", "country", "ingestion_time").show(5, truncate=False)

✅ Bronze stream durduruldu



26/05/10 13:57:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


📊 Bronze tabloda toplam: 10,100 satır

İlk 5 satır:
+------------+---------+---------+--------+--------------+-----------------------+
|kullanici_ID|olay_tipi|ilgili_ID|quantity|country       |ingestion_time         |
+------------+---------+---------+--------+--------------+-----------------------+
|13085       |purchase |489434   |12      |United Kingdom|2026-05-10 13:56:01.397|
|13085       |purchase |489434   |12      |United Kingdom|2026-05-10 13:56:01.397|
|13085       |purchase |489434   |12      |United Kingdom|2026-05-10 13:56:01.397|
|13085       |purchase |489434   |48      |United Kingdom|2026-05-10 13:56:01.397|
|13085       |purchase |489434   |24      |United Kingdom|2026-05-10 13:56:01.397|
+------------+---------+---------+--------+--------------+-----------------------+
only showing top 5 rows



In [10]:
# Olay tipi dağılımı
print("📊 Olay tipi dağılımı:")
df_bronze.groupBy("olay_tipi").count().show()

# Ülke dağılımı (top 10)
print("📊 Ülke dağılımı (top 10):")
df_bronze.groupBy("country").count().orderBy(col("count").desc()).show(10)

# Null değer kontrolü
from pyspark.sql.functions import col, sum as _sum, when
null_counts = df_bronze.select([
    _sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_bronze.columns
])
print("📊 Null değer sayıları:")
null_counts.show(vertical=True, truncate=False)

📊 Olay tipi dağılımı:
+------------+-----+
|   olay_tipi|count|
+------------+-----+
|    purchase| 9925|
|cancellation|  175|
+------------+-----+

📊 Ülke dağılımı (top 10):
+---------------+-----+
|        country|count|
+---------------+-----+
| United Kingdom| 9621|
|         France|  120|
|           EIRE|  119|
|       Portugal|   79|
|        Germany|   49|
|Channel Islands|   39|
|    Netherlands|   23|
|         Poland|   22|
|      Australia|   18|
|          Japan|    3|
+---------------+-----+
only showing top 10 rows

📊 Null değer sayıları:
-RECORD 0---------------
 kafka_topic     | 0    
 kafka_partition | 0    
 kafka_offset    | 0    
 kafka_timestamp | 0    
 timestamp       | 0    
 kullanici_ID    | 2805 
 olay_tipi       | 0    
 ilgili_ID       | 0    
 stock_code      | 0    
 description     | 16   
 quantity        | 0    
 unit_price      | 0    
 country         | 0    
 invoice_date    | 0    
 ingestion_time  | 0    



In [11]:
spark.stop()
print("✅ Spark Session kapatıldı")

✅ Spark Session kapatıldı
